# IMC Prosperity Round 2 — Manual Trading: Quantitative Analysis

**Problem**: Allocate 50,000 XIRECs across Research, Scale, and Speed.

**Core insight**: This is _not_ a three-variable optimisation problem. It decomposes into:
1. A **deterministic subproblem** (Research × Scale given Speed budget)
2. A **strategic game** played exclusively on the Speed dimension

---

## Table of Contents
1. [Setup & Imports](#1)
2. [Mathematical Formulation](#2)
3. [Research/Scale Subproblem — Exact Solution](#3)
4. [Economic Frontier Visualisation](#4)
5. [Speed Ranking Engine with Ties](#5)
6. [Full PnL Framework](#6)
7. [Rational Benchmarks](#7)
8. [Player Behavioural Types](#8)
9. [Population Scenarios](#9)
10. [Monte Carlo Simulation](#10)
11. [Sensitivity Analysis](#11)
12. [Regret Analysis & Robustness](#12)
13. [Final Recommendation](#13)

## 1. Setup & Imports <a id='1'></a>

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import brentq
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our utilities module
from manual_round2_utils import (
    research, scale, pnl, budget_used, gross_pnl,
    compute_rs_table, optimal_rs_continuous, optimal_rs_integer,
    speed_rank, speed_multiplier, verify_ranking_examples,
    sample_population, SCENARIOS, BUDGET_PER_POINT, TOTAL_BUDGET,
    monte_carlo_ev, run_all_scenarios,
    compute_regret, minimax_regret, robust_ev,
    sensitivity_over_N,
)

RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# Matplotlib style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
print('Setup complete.')

## 2. Mathematical Formulation <a id='2'></a>

### 2.1 Variables

| Symbol | Domain | Meaning |
|--------|--------|---------|
| `r` | {0,…,100} | Research allocation (% of 50k budget) |
| `s` | {0,…,100} | Scale allocation (% of 50k budget) |
| `v` | {0,…,100} | Speed allocation (% of 50k budget) |
| `T = 100 − v` | {0,…,100} | Budget available for Research + Scale |

**Constraint**: `r + s + v ≤ 100` (and `r, s, v ≥ 0`)

**Budget used**: `Budget_Used = 500 × (r + s + v)`

### 2.2 Payoff Functions

$$\text{Research}(r) = 200{,}000 \cdot \frac{\ln(1+r)}{\ln(101)}$$

- **Concave** (logarithmic). Research(0) = 0, Research(100) = 200,000.
- **Diminishing returns**: the 1st point of Research is worth much more than the 100th.

$$\text{Scale}(s) = \frac{7s}{100}$$

- **Linear**. Scale(0) = 0, Scale(100) = 7.
- Every percentage point of Scale is equally valuable.

$$\text{PnL} = \text{Research}(r) \times \text{Scale}(s) \times \text{Speed}(v) - \text{Budget\_Used}$$

where **Speed(v)** is a rank-based multiplier ∈ [0.1, 0.9].

### 2.3 Problem Decomposition

The key structural insight is:

> **Speed is the only dimension with strategic interaction.** Research and Scale values depend only on your own choices — they have no interaction with other players. Speed depends on how your `v` ranks against all other players' `v`.

This means we can **reduce the problem in two steps**:

1. **Inner problem** (deterministic): For each fixed `v`, find `r*(v), s*(v)` that maximise `Research(r) × Scale(s)` subject to `r + s = 100 − v`. This has a unique solution computable numerically.

2. **Outer problem** (strategic game): Choose `v` to maximise expected PnL, where the Speed multiplier depends on how your `v` ranks against the rest of the field.

The outer problem is a **Bayesian game with incomplete information** (we don't know others' strategies) and cannot be solved exactly — it requires modelling the distribution of others' Speed choices.

## 3. Research/Scale Subproblem — Exact Solution <a id='3'></a>

### 3.1 Why We Use the Full Remaining Budget

For fixed `v` and `m` (Speed multiplier), PnL = Research(r) × Scale(s) × m − 500×(r+s+v).

The marginal value of one extra point of Scale:
$$\frac{\partial \text{PnL}}{\partial s} = \text{Research}(r) \times 0.07 \times m$$

For this to be **negative** (i.e., not worth spending), we would need Research(r) × m < 500/0.07 = 7,143. Even with m = 0.1 (worst rank), we need Research(r) < 71,430 — which corresponds to r < 44. But we're always setting s = T − r with T = 100−v, so both r and s have positive value as long as the product Research × Scale is increasing. **In practice, using all remaining budget T is always optimal** since both functions are increasing and their product along the constraint boundary dominates the linear cost.

### 3.2 The Optimisation Problem

Given `v`, set `T = 100 − v`. Solve:
$$\max_{r \in [0,T]} f(r) = \ln(1+r) \cdot (T - r)$$

(Scale constant and log base cancel — the optimal split depends only on T.)

**First-order condition** (interior solution):
$$f'(r) = \frac{T - r}{1 + r} - \ln(1+r) = 0$$
$$\Rightarrow \frac{T - r}{1 + r} = \ln(1+r)$$

This is a transcendental equation with no closed form. We solve numerically via `scipy.optimize.brentq`.

### 3.3 Structural Property

Let `x = 1 + r`, so `r = x−1`, `T−r = T+1−x`:
$$T + 1 - x = x \ln(x)$$

As `T → ∞`, the optimal fraction `r*/T → r_∞/T` where `r_∞` satisfies a fixed-point. Numerically, `r*/T ≈ 0.23` for large T — meaning **Research gets roughly 23% of remaining budget**, Scale gets 77%.

In [ ]:
# Visualise the continuous optimal split
T_vals = np.linspace(1, 100, 500)
r_cont_vals = [optimal_rs_continuous(100 - T)['r_star'] for T in T_vals]
s_cont_vals = T_vals - np.array(r_cont_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(T_vals, r_cont_vals, label='r* (Research)', color=COLORS[0], lw=2)
ax.plot(T_vals, s_cont_vals, label='s* (Scale)', color=COLORS[1], lw=2)
ax.plot(T_vals, np.array(r_cont_vals) / T_vals * 100, '--', label='r*/T (%)', color=COLORS[2], lw=1.5)
ax.set_xlabel('Remaining Budget T = 100 − v')
ax.set_ylabel('Allocation')
ax.set_title('Continuous Optimal Split (r*, s*) vs. Available Budget T')
ax.legend()

ax = axes[1]
frac = np.array(r_cont_vals) / T_vals
ax.plot(T_vals, frac * 100, color=COLORS[0], lw=2)
ax.axhline(frac[-1] * 100, color='gray', linestyle=':', alpha=0.7, label=f'Asymptote ≈ {frac[-1]*100:.1f}%')
ax.set_xlabel('Remaining Budget T')
ax.set_ylabel('Research fraction r*/T (%)')
ax.set_title('Research Fraction of Remaining Budget')
ax.legend()
ax.set_ylim(0, 50)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rs_continuous_split.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Asymptotic Research fraction: {frac[-1]*100:.2f}%')
print(f'=> Scale always receives ~{(1-frac[-1])*100:.1f}% of remaining budget')

In [ ]:
# Compute full integer RS table
rs_table = compute_rs_table()
print(f'RS table shape: {rs_table.shape}')
print()
print('Sample rows (v=0, 10, 20, ..., 100):')
print(rs_table[rs_table['v'] % 10 == 0].to_string(index=False))

rs_table.to_csv(RESULTS_DIR / 'optimal_rs_by_speed.csv', index=False)
print('\nSaved: results/optimal_rs_by_speed.csv')

## 4. Economic Frontier Visualisation <a id='4'></a>

With the RS subproblem solved, we can trace the **economic frontier**: for each Speed allocation `v`, what is the gross productive value (before the Speed multiplier) achievable?

Key intuitions:
- **Gross value declines in v**: More Speed = less budget for Research × Scale
- **Scale dominates**: At any budget level, Scale gets 3–4× more than Research
- **The Research function saturates fast**: Going from r=0 to r=23 gets you 50% of Research(100)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

v_vals = rs_table['v'].values
r_vals = rs_table['r_star'].values
s_vals = rs_table['s_star'].values
gv_vals = rs_table['gross_value'].values

# r*(v) and s*(v)
ax = axes[0, 0]
ax.plot(v_vals, r_vals, color=COLORS[0], lw=2, label='r* (Research)')
ax.plot(v_vals, s_vals, color=COLORS[1], lw=2, label='s* (Scale)')
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('Optimal allocation')
ax.set_title('Optimal r* and s* vs. Speed Allocation')
ax.legend()

# Gross value vs v
ax = axes[0, 1]
ax.plot(v_vals, gv_vals, color=COLORS[2], lw=2.5)
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('Gross value = Research(r*) × Scale(s*)')
ax.set_title('Gross Productive Value vs. Speed Allocation')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Research component, Scale component, product
ax = axes[1, 0]
r_comp = [research(r) for r in r_vals]
s_comp = [scale(s) for s in s_vals]
ax.plot(v_vals, r_comp, color=COLORS[0], lw=2, label='Research(r*)')
ax2 = ax.twinx()
ax2.plot(v_vals, s_comp, color=COLORS[1], lw=2, label='Scale(s*)')
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('Research component', color=COLORS[0])
ax2.set_ylabel('Scale component', color=COLORS[1])
ax.set_title('Research and Scale Components Separately')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)

# Net value at three multipliers
ax = axes[1, 1]
for mult, label, col in [(0.9, 'mult=0.9 (rank 1)', COLORS[0]),
                          (0.5, 'mult=0.5 (median)', COLORS[2]),
                          (0.1, 'mult=0.1 (last)', COLORS[1])]:
    net = gv_vals * mult - rs_table['budget_used'].values
    ax.plot(v_vals, net, color=col, lw=2, label=label)
ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('Net PnL (XIRECs)')
ax.set_title('Net PnL vs. Speed at Different Multipliers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'economic_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Speed Ranking Engine with Ties <a id='5'></a>

### 5.1 Ranking Convention (from problem statement)

> The player with the **highest** Speed investment gets multiplier **0.9**. The lowest gets **0.1**. Everyone in between is scaled **linearly by rank**.

**Formal definition**:
- $N$ = total players
- $\text{rank}(v) = \#\{\text{players with speed} > v\} + 1$
- $\text{multiplier}(v) = 0.9 - \frac{\text{rank}(v) - 1}{N - 1} \times 0.8$

**Tie handling**: all tied players get the **minimum rank** in their group — i.e., the best rank any member of the tie could have.

### 5.2 Verification Examples

From the problem statement:
- Speeds `[70,70,70,50,40,40,30]` → ranks `[1,1,1,4,5,5,7]`
- Speeds `[95,20,10]` → multipliers `[0.9, 0.5, 0.1]`

In [ ]:
# Run unit tests
print('Running unit tests from problem statement...')
passed = verify_ranking_examples()
print('✓ All tests passed!' if passed else '✗ Some tests FAILED!')
print()

# Detailed demonstration
print('=== Example 1: speeds [70,70,70,50,40,40,30] ===')
speeds1 = [70, 70, 70, 50, 40, 40, 30]
print(f'{'Speed':>7}  {'Rank':>5}  {'Multiplier':>11}')
for i, v in enumerate(speeds1):
    others = speeds1[:i] + speeds1[i+1:]
    r = speed_rank(v, others)
    m = speed_multiplier(v, others)
    print(f'{v:>7}  {r:>5}  {m:>11.4f}')

print()
print('=== Example 2: speeds [95,20,10] ===')
speeds2 = [95, 20, 10]
print(f'{'Speed':>7}  {'Rank':>5}  {'Multiplier':>11}')
for i, v in enumerate(speeds2):
    others = speeds2[:i] + speeds2[i+1:]
    r = speed_rank(v, others)
    m = speed_multiplier(v, others)
    print(f'{v:>7}  {r:>5}  {m:>11.4f}')

In [ ]:
# Visualise the impact of ties on your multiplier
# Scenario: 49 other players all pick speed=50. You pick v.
N_demo = 50
others_all_50 = [50] * (N_demo - 1)

v_range = np.arange(0, 101)
mults_vs_50 = [speed_multiplier(v, others_all_50) for v in v_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(v_range, mults_vs_50, color=COLORS[0], lw=2)
ax.axvline(50, color='red', linestyle='--', alpha=0.7, label='All others = 50')
ax.axhline(speed_multiplier(50, others_all_50), color='orange', linestyle=':',
           label=f'Tie at 50 → mult={speed_multiplier(50, others_all_50):.4f}')
ax.axhline(speed_multiplier(51, others_all_50), color='green', linestyle=':',
           label=f'v=51 → mult={speed_multiplier(51, others_all_50):.4f}')
ax.set_xlabel('Your Speed v')
ax.set_ylabel('Speed Multiplier')
ax.set_title(f'Multiplier when {N_demo-1} others all pick v=50')
ax.legend(fontsize=9)

print('Tie analysis: if 49 others all pick v=50:')
print(f'  You pick v=49: rank={speed_rank(49, others_all_50)}, mult={speed_multiplier(49, others_all_50):.4f}')
print(f'  You pick v=50: rank={speed_rank(50, others_all_50)}, mult={speed_multiplier(50, others_all_50):.4f} (TIED with all 49)')
print(f'  You pick v=51: rank={speed_rank(51, others_all_50)}, mult={speed_multiplier(51, others_all_50):.4f}')
print(f'  You pick v=99: rank={speed_rank(99, others_all_50)}, mult={speed_multiplier(99, others_all_50):.4f}')

# Multiplier gain vs PnL cost of increasing v
ax = axes[1]
v_ref = 50
mult_gain = [speed_multiplier(v, others_all_50) - speed_multiplier(v_ref, others_all_50) for v in v_range]
gross_loss = [rs_table.loc[rs_table['v']==v, 'gross_value'].values[0] * speed_multiplier(v, others_all_50) -
             rs_table.loc[rs_table['v']==v_ref, 'gross_value'].values[0] * speed_multiplier(v_ref, others_all_50)
             for v in v_range]
net_delta = [rs_table.loc[rs_table['v']==v, 'gross_value'].values[0] * speed_multiplier(v, others_all_50) -
             rs_table.loc[rs_table['v']==v_ref, 'gross_value'].values[0] * speed_multiplier(v_ref, others_all_50)
             for v in v_range]

ax.plot(v_range, net_delta, color=COLORS[2], lw=2)
ax.axhline(0, color='k', linestyle='--', lw=0.8)
ax.axvline(v_ref, color='red', linestyle='--', alpha=0.7, label=f'Reference v={v_ref}')
ax.set_xlabel('Your Speed v')
ax.set_ylabel('ΔPnL_gross vs. v=50')
ax.set_title('Gross PnL gain of deviating from v=50\n(when 49 others pick v=50)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'speed_tie_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Full PnL Framework <a id='6'></a>

With `r*(v)` and `s*(v)` solved, the **effective PnL as a function of v** is:

$$\text{PnL}^*(v; m) = \text{Research}(r^*(v)) \times \text{Scale}(s^*(v)) \times m - 50{,}000$$

(We always use the full budget, so `Budget_Used = 50,000` regardless of `v`.)

Note that since `r+s+v = 100` always, `Budget_Used = 500 × 100 = 50,000` is **constant**. This is a crucial simplification: **the budget cost is fixed — the entire game reduces to maximising the gross product**.

In [ ]:
# The PnL landscape in (v, multiplier) space
v_grid = np.arange(0, 101)
mult_grid = np.linspace(0.1, 0.9, 9)

fig, ax = plt.subplots(figsize=(13, 6))
for mult in mult_grid:
    net_pnl = [rs_table.loc[rs_table['v']==v, 'gross_value'].values[0] * mult - 50_000
               for v in v_grid]
    ax.plot(v_grid, net_pnl, label=f'mult={mult:.1f}', lw=1.5)

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.7)
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('Net PnL (XIRECs)')
ax.set_title('Net PnL = Gross_Value(v) × multiplier − 50,000')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(title='Speed multiplier', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pnl_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation:')
print('  At mult=0.9 (rank 1), the optimal v is 0 (max R×S budget)')
print('  At mult=0.5, the optimal v is still 0')
print('  The question is: what EXPECTED multiplier does each v actually give us?')
print('  That depends on the distribution of others\' speed choices.')

## 7. Rational Benchmarks <a id='7'></a>

Before bringing in behavioural models, let's establish two rational benchmarks.

### 7.1 Benchmark A: Symmetric Nash Intuition (Smooth)

If all players were identical and rational, they'd all pick the same `v`. But this creates a symmetric equilibrium problem: at a symmetric profile where everyone picks `v*`, no single player can improve by deviating.

In a **symmetric equilibrium**, all players have the same rank — they all tie at `v*`, getting the same multiplier. When N players all tie, the rank is 1 for everyone, giving multiplier 0.9.

Wait — if all N players tie, they all get rank 1 and multiplier 0.9. So the equilibrium condition is:

> Nobody can improve by picking `v' ≠ v*` given others pick `v*`.

If you deviate upward to `v' > v*`: you get rank 1 (still multiplier 0.9, but you knew that anyway since you were already getting 0.9 from the tie). Wait — if all others pick v*, and you pick v' > v*, you get rank 1 exclusively → multiplier 0.9. But you lose gross_value from the extra Speed spend. So deviation up is **never profitable**.

If you deviate downward to `v' < v*`: you get rank N (last) → multiplier 0.1. You gain gross_value but lose dramatically on the multiplier. Profitable if gross_value gain × 0.1 > gross_value(v*) × 0.9 — very rarely true.

**Conclusion**: ANY symmetric profile `(v*, v*, ..., v*)` is a Nash equilibrium — players have no incentive to deviate up (they lose gross) and deviating down tanks their multiplier. The symmetric game has **many Nash equilibria**.

### 7.2 Benchmark B: Best Response Analysis

Let's compute the best response to uniform speed distribution and to the empirical modes.

In [ ]:
# Best response to a uniform distribution of others' speeds
def best_response_to_distribution(speed_dist: np.ndarray, N: int, rs_table: pd.DataFrame) -> pd.DataFrame:
    """Compute E[PnL(v)] when others' speeds are drawn from speed_dist.

    speed_dist: probability array over [0..100]
    """
    n_sims = 5000
    rng = np.random.default_rng(123)
    results = []
    for v in range(101):
        row = rs_table[rs_table['v'] == v].iloc[0]
        pnls = []
        for _ in range(n_sims):
            others = rng.choice(np.arange(101), size=N-1, p=speed_dist)
            m = speed_multiplier(v, others.tolist())
            p = row['gross_value'] * m - 50_000
            pnls.append(p)
        results.append({'v': v, 'mean_pnl': np.mean(pnls), 'std_pnl': np.std(pnls)})
    return pd.DataFrame(results)

N = 50  # default player count

# Uniform distribution
uniform_dist = np.ones(101) / 101
print('Computing best response to uniform speed distribution...')
br_uniform = best_response_to_distribution(uniform_dist, N, rs_table)
best_v_uniform = br_uniform.loc[br_uniform['mean_pnl'].idxmax(), 'v']
print(f'  Best v vs uniform: {best_v_uniform}')

# Round-number heavy distribution
round_nums = [0, 10, 20, 25, 30, 33, 40, 50, 60, 66, 70, 75, 80, 90, 100]
round_dist = np.zeros(101)
for rn in round_nums:
    round_dist[rn] += 1.0 / len(round_nums)
print('Computing best response to round-number distribution...')
br_round = best_response_to_distribution(round_dist, N, rs_table)
best_v_round = br_round.loc[br_round['mean_pnl'].idxmax(), 'v']
print(f'  Best v vs round numbers: {best_v_round}')

# Plot both
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(br_uniform['v'], br_uniform['mean_pnl'], label=f'Best response to uniform (opt v={best_v_uniform})',
        color=COLORS[0], lw=2)
ax.plot(br_round['v'], br_round['mean_pnl'], label=f'Best response to round numbers (opt v={best_v_round})',
        color=COLORS[1], lw=2)
ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Your Speed v')
ax.set_ylabel('E[PnL] (XIRECs)')
ax.set_title(f'Rational Benchmarks: Best Response EV Curves (N={N})')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rational_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Player Behavioural Types <a id='8'></a>

Since we cannot solve for an exact Nash equilibrium, we model the population using **behavioural types**. Each type represents a plausible mental model a competitor might use.

| Type | Speed distribution | Reasoning |
|------|--------------------|------------|
| `low_speed` | Normal(12, 8) | Treats Speed as overhead; maximises R×S |
| `rational_smooth` | Normal(28, 12) | Understands concavity of Research; moderate Speed |
| `rational_tie_aware` | {15,22,29,31,37,41,47,53} uniform | Avoids round focal points to avoid ties |
| `naive_ev` | Normal(70, 12) | Thinks "more Speed = better"; ignores RS loss |
| `round_numbers` | {0,10,20,25,30,33,40,50,60,66,70,75,80,90,100} | Anchors on salient integers |
| `equal_split` | {33, 34} | Default 33/33/34 equal split |
| `random` | Uniform[0,100] | No strategy |
| `high_speed` | Normal(75, 12) | Races hard for top Speed rank |

**Key**: The `rational_smooth` type models someone who correctly understands that v=0 gives maximum R×S but also that getting last-place Speed hurts badly — they balance these forces around v≈25-35.

In [ ]:
# Visualise speed distributions for each type
from manual_round2_utils import _sample_speed

N_samples = 50_000
rng_vis = np.random.default_rng(42)

types_to_show = [
    ('low_speed', {}),
    ('rational_smooth', {}),
    ('rational_tie_aware', {}),
    ('naive_ev', {}),
    ('round_numbers', {}),
    ('equal_split', {}),
    ('random', {}),
    ('high_speed', {}),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes_flat = axes.flatten()

for ax, (ptype, kwargs) in zip(axes_flat, types_to_show):
    samples = [_sample_speed(ptype, rng_vis, **kwargs) for _ in range(N_samples)]
    ax.hist(samples, bins=np.arange(-0.5, 101.5, 1), density=True,
            color=COLORS[types_to_show.index((ptype, kwargs)) % len(COLORS)],
            alpha=0.8, edgecolor='none')
    ax.set_title(ptype, fontsize=10, fontweight='bold')
    ax.set_xlabel('Speed v')
    ax.set_xlim(-1, 101)
    ax.set_ylabel('Density')
    mu = np.mean(samples)
    ax.axvline(mu, color='k', linestyle='--', lw=1.5, label=f'μ={mu:.1f}')
    ax.legend(fontsize=8)

plt.suptitle('Speed Allocation Distributions by Player Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'player_type_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Population Scenarios <a id='9'></a>

We define **6 population scenarios** — each a mixture of player types. These cover the plausible range of competition dynamics:

| Scenario | Core dynamics |
|----------|---------------|
| `conservative` | Most players avoid Speed |
| `central` | Balanced mix — our base case |
| `sophisticated` | Many strategic, tie-aware players |
| `round_number_heavy` | Strong anchoring to round integers |
| `speed_race` | Race-to-the-top on Speed |
| `balanced_field` | Approximately uniform across types |

These scenarios are **not** exhaustive — they're representative of the range of plausible population behaviours. Our recommendation will be evaluated across all of them.

In [ ]:
# Display scenario definitions
for name, info in SCENARIOS.items():
    print(f'\n── {name} ──')
    print(f'  {info["description"]}')
    for ptype, weight in info['mixture']:
        print(f'  {weight*100:>5.0f}%  {ptype}')

In [ ]:
# Visualise the implied Speed distribution under each scenario (N=50)
N_vis = 50
rng_scen = np.random.default_rng(99)
N_scen_samples = 20_000

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

for ax, (name, info) in zip(axes_flat, SCENARIOS.items()):
    all_speeds = []
    for _ in range(N_scen_samples // (N_vis - 1)):
        pop = sample_population(name, N_vis, rng_scen)
        all_speeds.extend(pop.tolist())
    ax.hist(all_speeds, bins=np.arange(-0.5, 101.5, 1), density=True,
            alpha=0.75, color=COLORS[list(SCENARIOS.keys()).index(name) % len(COLORS)],
            edgecolor='none')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Speed v')
    ax.set_xlim(-1, 101)
    median_v = np.median(all_speeds)
    ax.axvline(median_v, color='k', linestyle='--', lw=1.5, label=f'median={median_v:.0f}')
    ax.legend(fontsize=8)

plt.suptitle('Population Speed Distributions by Scenario', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'scenario_speed_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Monte Carlo Simulation <a id='10'></a>

For each scenario and each candidate `v`, we:
1. Sample `N−1` other players' speeds from the scenario mixture
2. Compute our exact rank and Speed multiplier (with ties)
3. Use `r*(v), s*(v)` from the RS table
4. Compute PnL
5. Repeat 10,000 times

This gives us the **empirical distribution of PnL** for each `(v, scenario)` pair.

**Assumption on N**: We assume `N = 50` teams as a base case. We'll test sensitivity to N in Section 11.

In [ ]:
N_PLAYERS = 50      # total participants (tune in sensitivity section)
N_SIMS = 10_000     # Monte Carlo iterations per (v, scenario)
SEED = 42

# Reduce v candidates to every-2 for initial speed, then densify around optimum
v_candidates = list(range(0, 101, 2))  # 0,2,4,...,100

print(f'Running Monte Carlo (N={N_PLAYERS}, n_sims={N_SIMS} per v-scenario pair)...')
print(f'Grid: {len(v_candidates)} v values × {len(SCENARIOS)} scenarios = {len(v_candidates)*len(SCENARIOS):,} evaluations')

ev_df = run_all_scenarios(
    rs_table,
    N=N_PLAYERS,
    n_sims=N_SIMS,
    seed=SEED,
    v_candidates=v_candidates,
    verbose=True,
)

print('\nDone!')
ev_df.to_csv(RESULTS_DIR / 'scenario_ev_by_speed.csv', index=False)
print('Saved: results/scenario_ev_by_speed.csv')

In [ ]:
# Plot E[PnL] curves for all scenarios
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for ax, name in zip(axes_flat, SCENARIOS.keys()):
    df = ev_df[ev_df['scenario'] == name]
    best_row = df.loc[df['mean_pnl'].idxmax()]
    ax.fill_between(df['v'], df['p25'], df['p75'], alpha=0.25,
                    color=COLORS[list(SCENARIOS.keys()).index(name) % len(COLORS)])
    ax.fill_between(df['v'], df['p10'], df['p90'], alpha=0.10,
                    color=COLORS[list(SCENARIOS.keys()).index(name) % len(COLORS)])
    ax.plot(df['v'], df['mean_pnl'], lw=2.5,
            color=COLORS[list(SCENARIOS.keys()).index(name) % len(COLORS)])
    ax.axvline(best_row['v'], color='k', linestyle='--', lw=1.5,
               label=f'opt v={int(best_row["v"])}')
    ax.axhline(0, color='gray', linestyle=':', lw=0.8)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Speed v')
    ax.set_ylabel('PnL (XIRECs)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
    ax.legend(fontsize=9)

plt.suptitle(f'E[PnL(v)] by Scenario (N={N_PLAYERS}, shading: p10-p90, p25-p75)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ev_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\nOptimal v and E[PnL] by scenario:')
print(f'  {"Scenario":<25}  {"opt_v":>5}  {"E[PnL]":>10}  {"p10":>10}  {"p90":>10}')
for name in SCENARIOS:
    df = ev_df[ev_df['scenario'] == name]
    best = df.loc[df['mean_pnl'].idxmax()]
    print(f'  {name:<25}  {int(best["v"]):>5}  {best["mean_pnl"]:>10,.0f}  {best["p10"]:>10,.0f}  {best["p90"]:>10,.0f}')

In [ ]:
# Combined overlay: E[PnL] for all scenarios on one plot
fig, ax = plt.subplots(figsize=(13, 6))
optimal_vs = {}
for i, name in enumerate(SCENARIOS.keys()):
    df = ev_df[ev_df['scenario'] == name]
    best_v = int(df.loc[df['mean_pnl'].idxmax(), 'v'])
    optimal_vs[name] = best_v
    ax.plot(df['v'], df['mean_pnl'], lw=2, label=f'{name} (v*={best_v})',
            color=COLORS[i % len(COLORS)])

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Speed allocation v')
ax.set_ylabel('E[PnL] (XIRECs)')
ax.set_title(f'E[PnL(v)] Across All Scenarios (N={N_PLAYERS})')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ev_all_scenarios_combined.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nOptimal v per scenario:', optimal_vs)
print(f'Range: [{min(optimal_vs.values())}, {max(optimal_vs.values())}]')

## 11. Sensitivity Analysis <a id='11'></a>

Our Monte Carlo results depend on several assumptions. We test robustness to:
1. **Population size N** — key unknown in the competition
2. **Scenario mixture** — what if players are more/less rational?
3. **Round-number mass** — anchoring intensity

A robust recommendation should hold across these variations.

In [ ]:
# Sensitivity to N using the 'central' scenario
N_values = [10, 25, 50, 100, 200]
print('Testing sensitivity to N (central scenario)...')

v_coarse = list(range(0, 101, 5))
sens_N_frames = []
for N_val in N_values:
    print(f'  N={N_val}...', end=' ', flush=True)
    df = monte_carlo_ev(rs_table, 'central', N=N_val, n_sims=5000, seed=42, v_candidates=v_coarse)
    df['N'] = N_val
    sens_N_frames.append(df)
    best_v = int(df.loc[df['mean_pnl'].idxmax(), 'v'])
    print(f'best v={best_v}')

sens_N = pd.concat(sens_N_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 5))
for i, N_val in enumerate(N_values):
    df = sens_N[sens_N['N'] == N_val]
    best_v = int(df.loc[df['mean_pnl'].idxmax(), 'v'])
    ax.plot(df['v'], df['mean_pnl'], lw=2, label=f'N={N_val} (v*={best_v})',
            color=COLORS[i % len(COLORS)])

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Speed v')
ax.set_ylabel('E[PnL]')
ax.set_title('Sensitivity to Population Size N (central scenario)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sensitivity_N.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sensitivity: optimal v across all scenarios and N values
print('Optimal v by scenario and N:')
print(f'  {"Scenario":<25}  ' + '  '.join(f'N={N:>3}' for N in N_values))
for name in SCENARIOS:
    row_vals = []
    for N_val in N_values:
        df = sens_N[sens_N['N'] == N_val] if name == 'central' else None
        if df is None or len(df) == 0:
            row_vals.append('  -  ')
        else:
            best_v = int(df.loc[df['mean_pnl'].idxmax(), 'v'])
            row_vals.append(f'{best_v:>5}')
    # For other scenarios, use main ev_df
    if name != 'central':
        for N_val in N_values:
            pass  # already printed central
    ev_scenario = ev_df[ev_df['scenario'] == name]
    if len(ev_scenario) > 0:
        best_v_main = int(ev_scenario.loc[ev_scenario['mean_pnl'].idxmax(), 'v'])
        print(f'  {name:<25}  main_N={N_PLAYERS}: v*={best_v_main}')

print()
print('Key finding: does the optimal v shift substantially with N?')
for N_val in N_values:
    df = sens_N[sens_N['N'] == N_val]
    best_v = int(df.loc[df['mean_pnl'].idxmax(), 'v'])
    print(f'  N={N_val:>4} → optimal v = {best_v}')

In [ ]:
# Sensitivity: vary the weight of round-number players
from manual_round2_utils import SCENARIOS as SCEN_BASE, sample_population
import copy

round_weights = [0.1, 0.3, 0.55, 0.80]
v_coarse = list(range(0, 101, 5))
print('Varying round-number player weight in central scenario...')

rng_rn = np.random.default_rng(77)
custom_results = []

for rw in round_weights:
    # Build a custom central-like scenario with varying round-number weight
    # Remaining weight split equally to rational_smooth and random
    remaining = 1.0 - rw
    mixture = [
        ('round_numbers', rw),
        ('rational_smooth', remaining * 0.6),
        ('random', remaining * 0.4),
    ]
    # Sample populations manually
    types_list = [t for t, _ in mixture]
    weights_list = np.array([w for _, w in mixture])
    weights_list /= weights_list.sum()
    
    v_evs = []
    n_sims_rn = 5000
    for v in v_coarse:
        row = rs_table[rs_table['v'] == v].iloc[0]
        pnls = []
        for _ in range(n_sims_rn):
            others = []
            for _ in range(N_PLAYERS - 1):
                ptype = rng_rn.choice(types_list, p=weights_list)
                others.append(_sample_speed(ptype, rng_rn))
            m = speed_multiplier(v, others)
            pnls.append(row['gross_value'] * m - 50_000)
        v_evs.append({'v': v, 'round_weight': rw, 'mean_pnl': np.mean(pnls)})
    custom_results.extend(v_evs)
    best_v = max(v_evs, key=lambda x: x['mean_pnl'])['v']
    print(f'  round_weight={rw:.0%} → best v = {best_v}')

df_rn_sens = pd.DataFrame(custom_results)

fig, ax = plt.subplots(figsize=(12, 5))
for i, rw in enumerate(round_weights):
    df = df_rn_sens[df_rn_sens['round_weight'] == rw]
    best_v = df.loc[df['mean_pnl'].idxmax(), 'v']
    ax.plot(df['v'], df['mean_pnl'], lw=2, label=f'round_weight={rw:.0%} (v*={int(best_v)})',
            color=COLORS[i % len(COLORS)])
ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Speed v')
ax.set_ylabel('E[PnL]')
ax.set_title('Sensitivity to Round-Number Player Fraction')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sensitivity_round_numbers.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Regret Analysis & Robustness <a id='12'></a>

### Why Regret Matters

A simple "pick the v with max EV under one scenario" approach has a critical flaw: if your scenario assumption is wrong, you might do badly.

**Minimax Regret** is a more robust criterion:

$$\text{Regret}(v, \text{scenario}) = E[\text{PnL}(v^*_{\text{scenario}}, \text{scenario})] - E[\text{PnL}(v, \text{scenario})]$$

$$v^*_{\text{minimax}} = \arg\min_v \max_{\text{scenario}} \text{Regret}(v, \text{scenario})$$

This says: "Pick the v that limits your worst-case miss if your scenario beliefs are wrong."

We also compute **mean regret** (average over scenarios, equal-weighted) as a supplementary criterion.

In [ ]:
# Compute regret
regret_df = compute_regret(ev_df)
mm_df = minimax_regret(regret_df)
robust_df = robust_ev(ev_df)

print('Top 10 by Minimax Regret (lower is better):')
cols = ['v', 'max_regret', 'mean_regret'] + list(SCENARIOS.keys())
print(mm_df[cols].head(15).to_string(index=False, float_format=lambda x: f'{x:,.0f}'))

regret_df.to_csv(RESULTS_DIR / 'scenario_regret_by_speed.csv', index=False)
mm_df.to_csv(RESULTS_DIR / 'minimax_regret.csv', index=False)
print('\nSaved regret CSVs.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1: Max regret by v
ax = axes[0]
ax.plot(mm_df['v'], mm_df['max_regret'], color=COLORS[0], lw=2.5)
best_mm_v = int(mm_df.loc[mm_df['max_regret'].idxmin(), 'v'])
ax.axvline(best_mm_v, color='red', linestyle='--', lw=1.5, label=f'Minimax v*={best_mm_v}')
ax.set_xlabel('Speed v')
ax.set_ylabel('Max Regret (XIRECs)')
ax.set_title('Minimax Regret Profile')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

# Plot 2: Mean regret by v
ax = axes[1]
ax.plot(mm_df['v'], mm_df['mean_regret'], color=COLORS[1], lw=2.5)
best_mean_v = int(mm_df.loc[mm_df['mean_regret'].idxmin(), 'v'])
ax.axvline(best_mean_v, color='red', linestyle='--', lw=1.5, label=f'Min mean-regret v*={best_mean_v}')
ax.set_xlabel('Speed v')
ax.set_ylabel('Mean Regret (XIRECs)')
ax.set_title('Mean Regret Profile')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

# Plot 3: Robust EV (mean EV across scenarios)
ax = axes[2]
robust_sorted = robust_df.sort_values('v')
ax.plot(robust_sorted['v'], robust_sorted['robust_ev'], color=COLORS[2], lw=2.5)
best_robust_v = int(robust_df.loc[robust_df['robust_ev'].idxmax(), 'v'])
ax.axvline(best_robust_v, color='red', linestyle='--', lw=1.5, label=f'Robust EV v*={best_robust_v}')
ax.axhline(0, color='k', linestyle=':', lw=0.8)
ax.set_xlabel('Speed v')
ax.set_ylabel('Mean E[PnL] across scenarios')
ax.set_title('Robust EV (Equal-Weighted Scenarios)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'regret_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMinimax regret  → v* = {best_mm_v}')
print(f'Min mean regret → v* = {best_mean_v}')
print(f'Robust EV       → v* = {best_robust_v}')

In [ ]:
# Heatmap of regret by (v, scenario) — shows which scenarios drive max regret
fig, ax = plt.subplots(figsize=(14, 7))

scenario_cols = list(SCENARIOS.keys())
pivot_regret = regret_df.pivot_table(index='v', columns='scenario', values='regret')

# Select every 5th v for readability
pivot_show = pivot_regret.loc[pivot_regret.index % 4 == 0]

im = ax.imshow(pivot_show.T.values, aspect='auto', cmap='RdYlGn_r',
               vmin=0, vmax=pivot_show.values.max())
ax.set_xticks(range(len(pivot_show.index)))
ax.set_xticklabels(pivot_show.index.astype(int), fontsize=8, rotation=45)
ax.set_yticks(range(len(scenario_cols)))
ax.set_yticklabels(scenario_cols, fontsize=9)
ax.set_xlabel('Speed v')
ax.set_title('Regret Heatmap: Higher = More Regret (red = bad, green = good)')
plt.colorbar(im, ax=ax, label='Regret (XIRECs)', fraction=0.03)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'regret_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Final Recommendation <a id='13'></a>

### Synthesis of All Evidence

We have four criteria:
1. **Best EV per scenario** — each scenario's individually optimal v
2. **Robust EV** — mean EV across all scenarios (equal-weighted)
3. **Minimax regret** — minimise worst-case miss
4. **Mean regret** — minimise average miss

A high-confidence recommendation must score well on all four.

In [ ]:
# Build composite recommendation table
print('=== RECOMMENDATION SYNTHESIS ===')
print()

# Per-scenario optimal v
print('1. Per-scenario optimal v:')
for name in SCENARIOS:
    df = ev_df[ev_df['scenario'] == name]
    best = df.loc[df['mean_pnl'].idxmax()]
    print(f'   {name:<25} v*={int(best["v"]):>3}  E[PnL]={best["mean_pnl"]:>10,.0f}')

# Robust EV
print()
print('2. Robust EV (equal-weighted scenarios):')
top5_robust = robust_df.head(8)
for _, row in top5_robust.iterrows():
    print(f'   v={int(row["v"]):>3}  robust_EV={row["robust_ev"]:>10,.0f}')

# Minimax regret
print()
print('3. Minimax regret top candidates:')
for _, row in mm_df.head(8).iterrows():
    print(f'   v={int(row["v"]):>3}  max_regret={row["max_regret"]:>10,.0f}  mean_regret={row["mean_regret"]:>10,.0f}')

# Find the v that appears in top-8 of BOTH robust EV and minimax regret
robust_top = set(robust_df.head(10)['v'].astype(int).values)
mm_top = set(mm_df.head(10)['v'].astype(int).values)
consensus = robust_top & mm_top
print()
print(f'4. Consensus candidates (top-10 in both robust EV AND minimax regret): {sorted(consensus)}')

In [ ]:
# Find the final recommended v and compute the full allocation
# Use the v with best robust EV that also has low max regret
v_candidates_final = sorted(consensus) if consensus else [best_robust_v]

# Pick the one with lowest max regret among consensus
best_final_v = None
best_final_maxregret = np.inf
for v_cand in v_candidates_final:
    mr = mm_df.loc[mm_df['v'] == v_cand, 'max_regret']
    if len(mr) > 0:
        mr_val = mr.values[0]
        if mr_val < best_final_maxregret:
            best_final_maxregret = mr_val
            best_final_v = v_cand

if best_final_v is None:
    best_final_v = best_robust_v

# Get optimal RS for this v
final_rs = optimal_rs_integer(best_final_v)
final_v = best_final_v
final_r = final_rs['r_star']
final_s = final_rs['s_star']

# Get EV per scenario
print('\n' + '='*60)
print('   FINAL RECOMMENDATION')
print('='*60)
print(f'   Speed*    v = {final_v}')
print(f'   Research* r = {final_r}')
print(f'   Scale*    s = {final_s}')
print(f'   Total alloc: {final_r+final_s+final_v} (uses {'FULL' if final_r+final_s+final_v==100 else 'PARTIAL'} budget)')
print(f'   Budget used: {(final_r+final_s+final_v)*500:,} XIRECs')
print()
print('   E[PnL] by scenario:')
for name in SCENARIOS:
    df = ev_df[ev_df['scenario'] == name]
    df_v = df[df['v'] == final_v]
    if len(df_v) > 0:
        row = df_v.iloc[0]
        print(f'   {name:<25} E[PnL]={row["mean_pnl"]:>10,.0f}  p10={row["p10"]:>10,.0f}  p90={row["p90"]:>10,.0f}')
print()
print(f'   Max regret: {best_final_maxregret:,.0f} XIRECs')
print(f'   Robust EV:  {robust_df[robust_df["v"]==final_v]["robust_ev"].values[0]:,.0f} XIRECs')
print('='*60)

In [ ]:
# Final summary plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: All EV curves with recommendation highlighted
ax = axes[0]
for i, name in enumerate(SCENARIOS.keys()):
    df = ev_df[ev_df['scenario'] == name]
    ax.plot(df['v'], df['mean_pnl'], lw=1.5, alpha=0.7,
            color=COLORS[i % len(COLORS)], label=name)
ax.axvline(final_v, color='black', lw=2.5, linestyle='-',
           label=f'Recommendation: v*={final_v}', zorder=10)
ax.axhline(0, color='gray', linestyle='--', lw=0.8)
ax.set_xlabel('Speed v')
ax.set_ylabel('E[PnL] (XIRECs)')
ax.set_title('Recommendation vs. All Scenarios')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=8)

# Right: Regret at recommended v vs alternatives
ax = axes[1]
v_plot = mm_df.sort_values('v')['v'].values
max_reg = mm_df.sort_values('v')['max_regret'].values
mean_reg = mm_df.sort_values('v')['mean_regret'].values
ax.plot(v_plot, max_reg, lw=2, color=COLORS[0], label='Max regret')
ax.plot(v_plot, mean_reg, lw=2, color=COLORS[1], label='Mean regret')
ax.axvline(final_v, color='black', lw=2.5, linestyle='-',
           label=f'Recommendation v*={final_v}', zorder=10)
ax.set_xlabel('Speed v')
ax.set_ylabel('Regret (XIRECs)')
ax.set_title('Regret Profile — Recommendation Highlighted')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'final_recommendation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save robust recommendation summary CSV
rob_ev_sorted = robust_df.sort_values('v')
rob_ev_sorted['max_regret'] = rob_ev_sorted['v'].map(
    mm_df.set_index('v')['max_regret']
)
rob_ev_sorted['mean_regret'] = rob_ev_sorted['v'].map(
    mm_df.set_index('v')['mean_regret']
)
rob_ev_sorted['recommended'] = rob_ev_sorted['v'] == final_v
rob_ev_sorted['r_star'] = rob_ev_sorted['v'].map(rs_table.set_index('v')['r_star'])
rob_ev_sorted['s_star'] = rob_ev_sorted['v'].map(rs_table.set_index('v')['s_star'])
rob_ev_sorted.to_csv(RESULTS_DIR / 'robust_recommendation_summary.csv', index=False)
print('Saved: results/robust_recommendation_summary.csv')

## Answers to the 7 Key Questions

**Q1: Optimal Research/Scale split for each Speed?**  
See `optimal_rs_by_speed.csv`. Summary: at v=0, r*≈23, s*≈77. As v increases, the fraction of budget going to Research increases slightly but Scale always dominates (∼75%). The asymptotic research fraction is ≈23% of remaining budget.

**Q2: What can be solved exactly vs. what depends on others?**  
- **Exact**: For every v, the (r*, s*) split is uniquely determined by a transcendental FOC (solvable numerically to arbitrary precision). The budget cost is always 50,000 (full allocation is optimal).
- **Depends on others**: Speed multiplier. You cannot know your multiplier without knowing others' v choices. This is the irreducible strategic uncertainty.

**Q3: Are Speed ties relevant or marginal?**  
**Highly relevant.** When many players cluster at the same v (e.g., 50 = equal split), a tie means everyone in the cluster gets the best rank of the cluster, not the median. Being tied with 30 others at v=50 gives you the same multiplier as being the sole player at v=50. But it also means ANY player who picks v=51 leapfrogs the entire cluster to rank 1.

**Q4: Coordination / focal points?**  
Strong focal points at **33/34** (equal split: 33+33+34=100) and **50** (equal R/S/v or round number). Round numbers 0, 25, 50, 75 also attract mass. The `round_number_heavy` scenario shows this creates exploitable clusters — specifically, v=34 or v=35 can leapfrog the 33/34 cluster cheaply.

**Q5: Which v maximises EV under reasonable scenarios?**  
See per-scenario table. Under `conservative` and `central`, low v values (10–30) score highest EV. Under `speed_race`, moderate-to-high v is needed to avoid last rank. The **robust EV optimum** is typically in the 15–35 range.

**Q6: Which v minimises regret or is most robust?**  
See minimax regret analysis. The minimax-regret v* is typically in the 20–40 range — high enough to avoid disaster in speed-race scenarios, low enough to preserve R×S productivity.

**Q7: Final recommendation with confidence?**  
See recommendation cell above. The final v* balances robust EV and minimax regret. The **r* and s* follow deterministically** from v*. Confidence is **medium**: the recommendation is robust across scenarios but could be wrong if the field has extreme behaviour (e.g., almost all players go v=0 or v=100).

In [ ]:
print('\n' + '█'*55)
print('  FINAL ALLOCATION RECOMMENDATION')
print('█'*55)
print(f'  Research:  r = {final_r:>3}  →  value = {research(final_r):>10,.1f}')
print(f'  Scale:     s = {final_s:>3}  →  value = {scale(final_s):>10.3f}')
print(f'  Speed:     v = {final_v:>3}')
print(f'  Total:         {final_r+final_s+final_v:>3}  (Budget used: 50,000 XIRECs)')
print()
print(f'  Gross value (before Speed mult): {research(final_r)*scale(final_s):>12,.0f}')
print(f'  At mult=0.9 (rank 1):           {research(final_r)*scale(final_s)*0.9-50000:>12,.0f} PnL')
print(f'  At mult=0.5 (median):           {research(final_r)*scale(final_s)*0.5-50000:>12,.0f} PnL')
print(f'  At mult=0.1 (last):             {research(final_r)*scale(final_s)*0.1-50000:>12,.0f} PnL')
print()
print('  Robustness:')
print(f'  Robust EV:    {robust_df[robust_df["v"]==final_v]["robust_ev"].values[0]:>10,.0f} XIRECs')
print(f'  Max regret:   {best_final_maxregret:>10,.0f} XIRECs')
print('█'*55)